In [2]:
'''3e) Implement an AI assistant using LangGraph or LangChain to answer queries'''

from pathlib import Path 

import os
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate

api_key = os.getenv("OPENAI_API_KEY")

base_url = (
    os.getenv("OPENAI_BASE_URL") or os.getenv("OPENAI_API_BASE")
)

if api_key is None:
    raise ValueError(
        "OPENAI_API_KEY was not found in the environment."
    )
print("OpenAI API key found")
if base_url:
    print("Custom OpenAI endpoint found")

with open("incident_knowledge_base.txt", "r", encoding="utf-8") as file: 
    knowledge_text = file.read()

documents = knowledge_text.split("================================================================================")
kb_documents = []

for document in documents: 
    document = document.strip()
    if document:
        kb_documents.append(document)

incidents = pd.read_csv("incident_reports.csv")
incident_documents = []

for _, row in incidents.iterrows():
    text = (
        f"Past incident {row['incident_id']}. "
        f"Type: {row['incident_type']}." 
        f"Severity: {row['severity']}."
        f"CPU: {row['cpu_utilization']}."
        f"Memory: {row['memory_usage']}."
        f"DB pool: {row['db_connection_pool_usage']}."
        f"Error rate: {row['application_error_rate']}."
        f"NEFT success: {row['neft_success_rate']}."
    )
    incident_documents.append(text)

documents = (kb_documents + incident_documents)

vectorizer = TfidfVectorizer(stop_words = "english")
document_vectors = vectorizer.fit_transform(documents)

def retrieve(query, top_k=3):
    query_vector = vectorizer.transform([query])
    scores = cosine_similarity(
        query_vector,
        document_vectors
    )[0]
    
    best_matches = scores.argsort()[
        ::-1
    ][:top_k]
    
    context = ""
    
    for index in best_matches:
        
        context +=documents[index]
        context +="\n\n"
    return context

print("\nLets Create Langchain Model")

model_name = os.getenv(
    "OPENAI_MODEL",
    "gpt-4o-mini"
)


llm_settings = {
    "model": model_name,
    "api_key": api_key
}

if base_url:
    llm_settings["base_url"] = base_url

llm = ChatOpenAI(
    **llm_settings
)

print("\nLets create the Incident Assistant Prompt")
prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        """You are an AI incident-analysis assistant for a digital bank. Use the retrieved information to answer the user's question. 
        Give: 
        1. Likely Issue
        2. Possible Root Cause
        3. Recommended Action
        4. Conclusion 
        5. Future analysis that may need to be looked at or investigated.
        
        Keep the answer short and practical and concise and straight to the point. 
        
        If the retrieved information is insufficient, say that more investigation is required and necessary. 
        
        Retrieved information 
        {context}
        """
    ), 
    (
        "human",
        "{question}"
    )
])
        
chain = prompt | llm        

#create a function to answer the questions via prompt

def answer_query(question):
    context = retrieve(question)
    response = chain.invoke({
        "context": context,
        "question": question
    })
    return response.content

question = (
    "NEFT transactions are failing with gateway timeout. What could be the cause? How should we fix it? Please provide more techniques/methods to fixing the issue."
)

answer = answer_query(
    question
)

print("\nYour Question:")
print(question)

print("\nAI Assistant Response Answer:")
print(answer)

#Example of Interactive Chat with AI Assistant
while True:
    question = input(
        "\nPlease ask a question about an incident (or type exit if you want to stop):"
    )
    
    if question.lower() == "exit":
        break
    
    answer = answer_query(
        question
    )
    
    print("\nAssistant:")
    print(answer)
    
    
    
    
    
        



        
        
    
    
    
    
    
    
    
        
        
        
        
        

OpenAI API key found
Custom OpenAI endpoint found

Lets Create Langchain Model

Lets create the Incident Assistant Prompt

Your Question:
NEFT transactions are failing with gateway timeout. What could be the cause? How should we fix it? Please provide more techniques/methods to fixing the issue.

AI Assistant Response Answer:
1. **Likely Issue**: NEFT transactions are failing due to payment gateway timeouts or unavailability.

2. **Possible Root Cause**: This issue may stem from external network connectivity problems, high latency, or a temporary service disruption from the payment gateway provider.

3. **Recommended Action**:
   - Check the payment gateway status and connectivity.
   - Review network logs for any connectivity issues.
   - Implement retry logic with exponential backoff to mitigate temporary outages.
   - Enable transaction monitoring and alerting to catch issues proactively.

4. **Conclusion**: Immediate focus should be on verifying the payment gateway's operational st


Please ask a question about an incident (or type exit if you want to stop): NEFT transactions are failing with gateway timeout. What could be the cause? How should we fix it? Please provide more techniques/methods to fixing the issue.



Assistant:
1. **Likely Issue**: NEFT transaction failures due to payment gateway timeouts.

2. **Possible Root Cause**: This can be attributed to payment gateway unavailability or network connectivity issues with external systems.

3. **Recommended Action**:
   - **Check Payment Gateway Status**: Assess the current status and availability of the payment gateway.
   - **Verify Beneficiary Details**: Ensure that the beneficiary account details are valid and properly configured.
   - **Review Network Logs**: Investigate network connectivity logs for any disruptions or issues that may affect communication with external systems.
   - **Implement Retry Logic**: Use an exponential backoff strategy for retrying transactions that fail due to timeouts.
   - **Enable Transaction Monitoring**: Set up monitoring and alerting for transaction failures to detect issues proactively.

4. **Conclusion**: The immediate focus should be on checking the payment gateway and network connectivity, while also e


Please ask a question about an incident (or type exit if you want to stop): who are you?



Assistant:
I am an AI incident-analysis assistant for a digital bank, here to help analyze incidents and provide insights based on the information available. How can I assist you today?



Please ask a question about an incident (or type exit if you want to stop): tell me about New York City's weather



Assistant:
More investigation is required to address your request as it does not pertain to the incident analysis for the digital bank. Please provide a question related to the retrieved incident information for assistance.



Please ask a question about an incident (or type exit if you want to stop): exit
